## Set Library Path

In [1]:
.libPaths("/home/vhaghani/anaconda3/envs/jupyter_nb_R4.4.2/lib/R/library")

## Load Libraries

In [3]:
library(UpSetR)
library(openxlsx)
library(readxl)
library(dplyr)
library(glue)
library(ggplot2)
library(viridis)
library(ggrepel)

## Visualize DEG Analysis

### Read in Data

In [4]:
# List all DEG files
files <- list.files(path = "05_DEGs", pattern = "\\_results.csv$", full.names = TRUE)

# View
head(files)

[1] "05_DEGs/R270X_D18_vs_R270X_D46_results.csv"
[2] "05_DEGs/R270X_D18_vs_R270X_D90_results.csv"
[3] "05_DEGs/R270X_D18_vs_WT_D18_results.csv"   
[4] "05_DEGs/R270X_D18_vs_WT_D46_results.csv"   
[5] "05_DEGs/R270X_D18_vs_WT_D90_results.csv"   
[6] "05_DEGs/R270X_D46_vs_R270X_D90_results.csv"

### Separate Upregulated and Downregulated Genes

In [5]:
# Save upregulated and downregulated gene lists
for (file in files) {
  # Extract the base filename without extension
  filename <- tools::file_path_sans_ext(basename(file))
  
  # Split the filename into parts based on '_vs_'
  split_name <- unlist(strsplit(filename, "_vs_"))
  
  # Handle different naming conventions and assign group1 and group2
  if (length(split_name) == 2) {
    group1 <- tolower(split_name[1])  
    group2 <- tolower(gsub("_results", "", split_name[2]))  
  }

  # Read the current file
  df <- read.csv(file)
  colnames(df)[1] <- "gene"

  # Filter to only significant DEGs
  df <- df[df$adj.P.Val < 0.05, ]

  # Filter for only upregulated genes
  upregulated_genes <- df[df$logFC > 0, "gene"]

  # Filter for only downregulated genes
  downregulated_genes <- df[df$logFC < 0, "gene"]

  # Write the upregulated genes to a file in the output directory
  upregulated_file <- file.path(paste0("05_DEGs/", group1, "_vs_", group2, "_upregulated.txt"))
  write.table(upregulated_genes, file = upregulated_file, row.names = FALSE, col.names = FALSE, quote = FALSE)

  # Write the downregulated genes to a file in the output directory
  downregulated_file <- file.path(paste0("05_DEGs/", group1, "_vs_", group2, "_downregulated.txt"))
  write.table(downregulated_genes, file = downregulated_file, row.names = FALSE, col.names = FALSE, quote = FALSE)
  
  # Print status
  cat("Processed", filename, "- Up:", length(upregulated_genes), "Down:", length(downregulated_genes), "\n")
}

Processed R270X_D18_vs_R270X_D46_results - Up: 4179 Down: 5096 
Processed R270X_D18_vs_R270X_D90_results - Up: 5655 Down: 6070 
Processed R270X_D18_vs_WT_D18_results - Up: 50 Down: 80 
Processed R270X_D18_vs_WT_D46_results - Up: 4225 Down: 5090 
Processed R270X_D18_vs_WT_D90_results - Up: 5470 Down: 6092 
Processed R270X_D46_vs_R270X_D90_results - Up: 2321 Down: 2579 
Processed R270X_D46_vs_WT_D46_results - Up: 48 Down: 43 
Processed R270X_D46_vs_WT_D90_results - Up: 2256 Down: 2704 
Processed R270X_D90_vs_WT_D90_results - Up: 33 Down: 33 
Processed WT_D18_vs_R270X_D46_results - Up: 4264 Down: 5158 
Processed WT_D18_vs_R270X_D90_results - Up: 5644 Down: 6060 
Processed WT_D18_vs_WT_D46_results - Up: 4304 Down: 5134 
Processed WT_D18_vs_WT_D90_results - Up: 5416 Down: 6059 
Processed WT_D46_vs_R270X_D90_results - Up: 3182 Down: 3466 
Processed WT_D46_vs_WT_D90_results - Up: 2915 Down: 3443 


### Make Upset Plot

In [6]:
# Read in gene list files
files <- list.files(path = "05_DEGs", pattern = "_(upregulated|downregulated)\\.txt$", full.names = TRUE)

# Create label mapping automatically
label_mapping <- setNames(
  sapply(files, function(file) {
    # Get just the filename without path
    filename <- basename(file)
    
    # Remove file extension
    name_without_ext <- tools::file_path_sans_ext(filename)
    
    # Apply transformation rules
    label <- name_without_ext %>%
      gsub("mecp2_", "", .) %>%        # Remove "mecp2_" prefix
      gsub("_cto_", " ", .) %>%         # Replace "_cto_" with space
      gsub("_downregulated$", " Down", .) %>%  # Replace _downregulated with " Down"
      gsub("_upregulated$", " Up", .) %>%      # Replace _upregulated with " Up"
      gsub("d(\\d+)", "D\\1", .) %>%    # Capitalize D in D111, D90, etc.
      gsub("del", "Del", .) %>%           # Capitalize d in "del"
      gsub("r270x", "R270X", .) %>%       # Capitalize r and x in "r270x"
      gsub("pa", "pA", .) %>%           # Capitalize p in "pa"
      gsub("pd", "pD", .) %>%            # Capitalize p in "pd"
      gsub("wt", "WT", .) %>%            # Capitalize WT
      gsub("_", " ", .)                 # Replace remaining underscores with spaces
    
    return(label)
  }),
  tools::file_path_sans_ext(basename(files)) 
)

# Print the resulting label mapping
print(label_mapping)

r270x_d18_vs_r270x_d46_downregulated   r270x_d18_vs_r270x_d46_upregulated 
       "R270X D18 vs R270X D46 Down"          "R270X D18 vs R270X D46 Up" 
r270x_d18_vs_r270x_d90_downregulated   r270x_d18_vs_r270x_d90_upregulated 
       "R270X D18 vs R270X D90 Down"          "R270X D18 vs R270X D90 Up" 
   r270x_d18_vs_wt_d18_downregulated      r270x_d18_vs_wt_d18_upregulated 
          "R270X D18 vs WT D18 Down"             "R270X D18 vs WT D18 Up" 
   r270x_d18_vs_wt_d46_downregulated      r270x_d18_vs_wt_d46_upregulated 
          "R270X D18 vs WT D46 Down"             "R270X D18 vs WT D46 Up" 
   r270x_d18_vs_wt_d90_downregulated      r270x_d18_vs_wt_d90_upregulated 
          "R270X D18 vs WT D90 Down"             "R270X D18 vs WT D90 Up" 
r270x_d46_vs_r270x_d90_downregulated   r270x_d46_vs_r270x_d90_upregulated 
       "R270X D46 vs R270X D90 Down"          "R270X D46 vs R270X D90 Up" 
   r270x_d46_vs_wt_d46_downregulated      r270x_d46_vs_wt_d46_upregulated 
          "R270X D46 vs W

In [7]:
# Initialize an empty list to store gene names
gene_list <- list()

# Loop through each file and read the gene names
for (file in files) {
  # Check if the file exists and is not empty
  if (file.exists(file) && file.info(file)$size > 0) {
    tryCatch({
      # Read the gene names from the file
      gene_names <- read.table(file, stringsAsFactors = FALSE, header = FALSE)$V1
      
      # Remove any empty strings or NA values
      gene_names <- gene_names[!is.na(gene_names) & gene_names != ""]
      
      # Only add to list if there are valid genes
      if (length(gene_names) > 0) {
        # Use the filename (without path and extension) as the list name
        list_name <- tools::file_path_sans_ext(basename(file))
        
        # Store the gene names in the list
        gene_list[[list_name]] <- unique(gene_names)  # Remove duplicates
        message(paste("Loaded", length(gene_names), "genes from", file))
      } else {
        message(paste("No valid genes found in:", file))
      }
    }, error = function(e) {
      message(paste("Error reading file:", file, "-", e$message))
    })
  } else {
    message(paste("Skipping empty or non-existent file:", file))
  }
}

Loaded 5096 genes from 05_DEGs/r270x_d18_vs_r270x_d46_downregulated.txt

Loaded 4179 genes from 05_DEGs/r270x_d18_vs_r270x_d46_upregulated.txt

Loaded 6070 genes from 05_DEGs/r270x_d18_vs_r270x_d90_downregulated.txt

Loaded 5655 genes from 05_DEGs/r270x_d18_vs_r270x_d90_upregulated.txt

Loaded 80 genes from 05_DEGs/r270x_d18_vs_wt_d18_downregulated.txt

Loaded 50 genes from 05_DEGs/r270x_d18_vs_wt_d18_upregulated.txt

Loaded 5090 genes from 05_DEGs/r270x_d18_vs_wt_d46_downregulated.txt

Loaded 4225 genes from 05_DEGs/r270x_d18_vs_wt_d46_upregulated.txt

Loaded 6092 genes from 05_DEGs/r270x_d18_vs_wt_d90_downregulated.txt

Loaded 5470 genes from 05_DEGs/r270x_d18_vs_wt_d90_upregulated.txt

Loaded 2579 genes from 05_DEGs/r270x_d46_vs_r270x_d90_downregulated.txt

Loaded 2321 genes from 05_DEGs/r270x_d46_vs_r270x_d90_upregulated.txt

Loaded 43 genes from 05_DEGs/r270x_d46_vs_wt_d46_downregulated.txt

Loaded 48 genes from 05_DEGs/r270x_d46_vs_wt_d46_upregulated.txt

Loaded 2704 genes from 0

In [8]:
# Check if we have any data to plot
if (length(gene_list) == 0) {
  stop("No gene data found in any of the files. Cannot create UpSet plot.")
}

message(paste("Successfully loaded data from", length(gene_list), "files"))

Successfully loaded data from 30 files



In [9]:
# Convert the gene list to a data frame suitable for UpSetR
gene_df <- data.frame(
  Gene = unlist(gene_list),
  Group = rep(names(gene_list), times = sapply(gene_list, length))
)

# View
head(gene_df)

,Gene,Group
,<chr>,<chr>
r270x_d18_vs_r270x_d46_downregulated1,P42857,r270x_d18_vs_r270x_d46_downregulated
r270x_d18_vs_r270x_d46_downregulated2,Q8N126,r270x_d18_vs_r270x_d46_downregulated
r270x_d18_vs_r270x_d46_downregulated3,Q16352,r270x_d18_vs_r270x_d46_downregulated
r270x_d18_vs_r270x_d46_downregulated4,Q9NZR1,r270x_d18_vs_r270x_d46_downregulated
r270x_d18_vs_r270x_d46_downregulated5,Q96E17,r270x_d18_vs_r270x_d46_downregulated
r270x_d18_vs_r270x_d46_downregulated6,P61925,r270x_d18_vs_r270x_d46_downregulated


In [10]:
# Create a binary presence-absence matrix for UpSetR
gene_matrix <- as.data.frame.matrix(table(gene_df))

In [11]:
# If label_mapping exists, rename the columns
if (exists("label_mapping")) {
  # Only rename columns that exist in the mapping
  common_cols <- intersect(colnames(gene_matrix), names(label_mapping))
  if (length(common_cols) > 0) {
    colnames(gene_matrix)[colnames(gene_matrix) %in% common_cols] <- label_mapping[common_cols]
  }
}

In [12]:
pdf("05_DEGs/upset_plot_of_DEGs.pdf", width = 17, height = 13)

# Create the UpSet plot
p <- upset(gene_matrix, sets = colnames(gene_matrix), keep.order = TRUE,
      main.bar.color = "steelblue", 
      sets.bar.color = "darkgreen",
      order.by = "freq", 
      text.scale = 0.75)

print(p)

dev.off()

pdf 
  2

### View Significantly Overlapping Genes

In [13]:
# Create gene membership from the binary matrix
gene_membership <- list()

for (i in 1:nrow(gene_matrix)) {
  gene_name <- rownames(gene_matrix)[i]
  present_conditions <- colnames(gene_matrix)[gene_matrix[i, ] == 1]
  
  if (length(present_conditions) > 0) {
    gene_membership[[gene_name]] <- present_conditions
  }
}

# Continue with the rest of the report code...
overlap_df <- data.frame(
  Gene = names(gene_membership),
  Condition_Count = sapply(gene_membership, length),
  Conditions = sapply(gene_membership, function(x) paste(sort(x), collapse = ", ")),
  stringsAsFactors = FALSE
)

# Sort by number of conditions (most overlapping first), then by gene name
overlap_df <- overlap_df[order(-overlap_df$Condition_Count, overlap_df$Gene), ]

# Generate text report
sink("05_DEGs/gene_overlap_report.txt")

cat("GENE OVERLAP ANALYSIS REPORT\n")
cat("============================\n\n")
cat("Generated on:", date(), "\n")
cat("Total unique genes:", length(gene_membership), "\n")
cat("Total conditions:", ncol(gene_matrix), "\n\n")

cat("CONDITIONS ANALYZED:\n")
cat(paste(colnames(gene_matrix), collapse = ", "), "\n\n")

cat("MOST OVERLAPPED GENES\n")
cat("=====================\n\n")

# Print genes grouped by overlap count
current_count <- -1
for (i in 1:nrow(overlap_df)) {
  if (overlap_df$Condition_Count[i] != current_count) {
    cat("\n", paste(rep("=", 60), collapse = ""), "\n", sep = "")
    cat("GENES APPEARING IN", overlap_df$Condition_Count[i], "CONDITIONS\n")
    cat(paste(rep("=", 60), collapse = ""), "\n\n", sep = "")
    current_count <- overlap_df$Condition_Count[i]
  }
  
  cat(overlap_df$Gene[i], "\n")
  cat("  Conditions: ", overlap_df$Conditions[i], "\n\n", sep = "")
}

sink()

# Save as CSV
write.csv(overlap_df, "05_DEGs/gene_overlap_detailed.csv", row.names = FALSE)

print(paste("Text report saved as: 05_DEGs/gene_overlap_report.txt"))
print(paste("CSV file saved: gene_overlap_detailed.csv"))

[1] "Text report saved as: 05_DEGs/gene_overlap_report.txt"
[1] "CSV file saved: gene_overlap_detailed.csv"
